# Unified Sanity Check Analysis

**Purpose:** Comprehensive data validation and visualization for time series forecasting project

**Analysis Components:**
1. Data Loading & Basic Info
2. Missing Values Analysis
3. Feature Groups Analysis (ULTRA/HIGH/CORE)
4. Correlation Analysis (Spearman/Pearson)
5. Distribution Analysis
6. Outlier Detection
7. Interactive Time Series Explorer
8. Interactive Distribution Explorer
9. Weight Analysis
10. Feature Importance (IC Analysis)

**Project Integration:**
- Validates data quality for LGBM SHAP models
- Checks feature engineering pipeline integrity
- Identifies data issues before model training
- Provides interactive exploration capabilities

## STEP 0: CONFIGURATION & SAMPLE PREPARATION

**Note:** Using sample data for widgets to ensure performance. Full dataset analysis available in separate cells.

In [ ]:
# ============================================
# IMPORTS & CONFIGURATION
# ============================================
import polars as pl
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ipywidgets import Dropdown, IntSlider, Button, Output, VBox, HBox, Checkbox, FloatText
from IPython.display import display
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('default')
sns.set_palette("husl")

# Project paths
from pathlib import Path
project_root = Path("..")
cleaned_data_dir = project_root / "data/cleaned"
sample_data_dir = project_root / "data/downloaded_data"

print("="*60)
print("UNIFIED SANITY CHECK ANALYSIS")
print("="*60)
print(f"Project root: {project_root}")
print(f"Cleaned data: {cleaned_data_dir}")
print(f"Sample data: {sample_data_dir}")

In [ ]:
# ============================================
# FEATURE GROUPS DEFINITION
# ============================================
FEATURE_GROUPS = {
    "ultra": ['feature_bz', 'feature_aw', 'feature_cc'],
    "high": ['feature_az', 'feature_bl', 'feature_l', 'feature_m'],
    "test38": ['feature_w', 'feature_x', 'feature_y', 'feature_z'],
    "core": ['feature_at', 'feature_by', 'feature_ay', 'feature_cd', 
             'feature_ce', 'feature_cf', 'feature_al'],
    "low": []  # Will be populated dynamically
}

print("="*60)
print("FEATURE GROUPS DEFINITION")
print("="*60)
for group, features in FEATURE_GROUPS.items():
    if features:
        print(f"{group.upper():8s} ({len(features):2d}): {features}")
    else:
        print(f"{group.upper():8s} ( 0): Will be populated dynamically")

In [ ]:
# ============================================
# DATA LOADING (SAMPLE FOR WIDGETS)
# ============================================
print("="*60)
print("LOADING SAMPLE DATA FOR WIDGETS")
print("="*60)

# Try sample data first, fallback to cleaned data
sample_paths = [
    sample_data_dir / "train_sample.parquet",
    cleaned_data_dir / "train_clean.parquet"
]

sample_loaded = False
for path in sample_paths:
    if path.exists():
        try:
            train_sample = pl.read_parquet(path)
            print(f"\nLoaded sample from: {path}")
            print(f"Sample shape: {train_sample.shape}")
            
            # Further sample if too large for widgets
            if len(train_sample) > 100000:
                train_sample = train_sample.sample(n=100000, seed=42)
                print(f"Downsampled to: {train_sample.shape}")
            
            sample_loaded = True
            break
        except Exception as e:
            print(f"Failed to load {path}: {e}")
            continue

if not sample_loaded:
    print("\nNo sample data available. Creating synthetic sample for widget testing...")
    # Create minimal synthetic data for widget testing
    n_rows = 10000
    train_sample = pl.DataFrame({
        'id': range(n_rows),
        'code': np.random.choice(['A', 'B', 'C'], n_rows),
        'sub_code': np.random.choice(['X', 'Y'], n_rows),
        'sub_category': np.random.choice(['CAT1', 'CAT2'], n_rows),
        'horizon': np.random.choice([1, 3, 10, 25], n_rows),
        'ts_index': np.random.randint(0, 3600, n_rows),
        'y_target': np.random.normal(0, 10, n_rows),
        'weight': np.random.exponential(1, n_rows),
    })
    
    # Add some feature columns
    for i in range(10):
        train_sample = train_sample.with_columns(
            pl.Series(f'feature_{chr(97+i)}', np.random.normal(0, 1, n_rows))
        )
    
    print(f"Created synthetic sample: {train_sample.shape}")

print(f"\nFinal sample for widgets: {train_sample.shape}")
print(f"Columns: {train_sample.columns}")

## STEP 1: BASIC DATA ANALYSIS

**Purpose:** Understand data structure, types, and basic statistics

In [ ]:
# ============================================
# BASIC DATA OVERVIEW
# ============================================
print("="*60)
print("BASIC DATA OVERVIEW")
print("="*60)

print(f"Shape: {train_sample.shape}")
print(f"Memory usage: {train_sample.estimated_size('mb'):.2f} MB")

# Column types
print("\nCOLUMN TYPES:")
for col in train_sample.columns:
    dtype = train_sample[col].dtype
    print(f"  {col:20s}: {dtype}")

# Basic statistics
print("\nBASIC STATISTICS:")
numeric_cols = [col for col in train_sample.columns if train_sample[col].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64]]
if numeric_cols:
    basic_stats = train_sample.select(numeric_cols).describe()
    print(basic_stats)

# Unique values in categorical columns
print("\nCATEGORICAL UNIQUE VALUES:")
cat_cols = ['code', 'sub_code', 'sub_category', 'horizon']
for col in cat_cols:
    if col in train_sample.columns:
        unique_vals = train_sample[col].unique()
        print(f"  {col:15s}: {len(unique_vals)} unique values")
        if len(unique_vals) <= 10:
            print(f"    Values: {unique_vals.to_list()}")

## STEP 1.1: INTERACTIVE DISTRIBUTION EXPLORER

**Purpose:** Interactive exploration of feature distributions by horizon and groups

In [ ]:
# ============================================
# INTERACTIVE DISTRIBUTION EXPLORER
# ============================================
print("="*60)
print("INTERACTIVE DISTRIBUTION EXPLORER")
print("="*60)

# Get feature columns
feature_cols = [col for col in train_sample.columns if col.startswith('feature_')]
print(f"Available features: {len(feature_cols)}")

# Create widgets
feature_dropdown = Dropdown(
    options=feature_cols,
    value=feature_cols[0] if feature_cols else None,
    description='Feature:',
    style={'description_width': 'initial'}
)

horizon_dropdown = Dropdown(
    options=['All'] + sorted(train_sample['horizon'].unique().to_list()),
    value='All',
    description='Horizon:',
    style={'description_width': 'initial'}
)

bins_slider = IntSlider(
    min=10,
    max=100,
    value=30,
    description='Bins:',
    style={'description_width': 'initial'}
)

log_scale_checkbox = Checkbox(
    value=False,
    description='Log Scale'
)

output_area = Output()

def plot_distribution(change=None):
    with output_area:
        output_area.clear_output()
        
        feature = feature_dropdown.value
        horizon = horizon_dropdown.value
        bins = bins_slider.value
        log_scale = log_scale_checkbox.value
        
        if feature is None:
            print("Please select a feature")
            return
        
        # Filter data
        if horizon != 'All':
            data = train_sample.filter(pl.col('horizon') == horizon)
        else:
            data = train_sample
        
        feature_data = data[feature].drop_nulls().to_numpy()
        
        if len(feature_data) == 0:
            print(f"No data for feature {feature} with horizon {horizon}")
            return
        
        # Create plot
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        
        # Histogram
        ax1.hist(feature_data, bins=bins, alpha=0.7, edgecolor='black')
        ax1.set_title(f'Distribution of {feature}\nHorizon: {horizon} (n={len(feature_data)})')
        ax1.set_xlabel(feature)
        ax1.set_ylabel('Frequency')
        if log_scale:
            ax1.set_yscale('log')
        ax1.grid(True, alpha=0.3)
        
        # Box plot
        ax2.boxplot(feature_data, vert=True)
        ax2.set_title(f'Box Plot of {feature}')
        ax2.set_ylabel(feature)
        ax2.grid(True, alpha=0.3)
        
        # Statistics
        stats_text = f"""
        Statistics:
        Count: {len(feature_data):,}
        Mean: {np.mean(feature_data):.4f}
        Std: {np.std(feature_data):.4f}
        Min: {np.min(feature_data):.4f}
        Max: {np.max(feature_data):.4f}
        Median: {np.median(feature_data):.4f}
        Skewness: {stats.skew(feature_data):.4f}
        Kurtosis: {stats.kurtosis(feature_data):.4f}
        """
        
        plt.figtext(0.02, 0.95, stats_text, fontsize=9, verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.tight_layout()
        plt.show()

# Connect widgets
feature_dropdown.observe(plot_distribution, names='value')
horizon_dropdown.observe(plot_distribution, names='value')
bins_slider.observe(plot_distribution, names='value')
log_scale_checkbox.observe(plot_distribution, names='value')

# Display widgets
controls = VBox([HBox([feature_dropdown, horizon_dropdown]), 
                HBox([bins_slider, log_scale_checkbox])])
display(controls, output_area)

# Initial plot
plot_distribution()

## STEP 1.2: INTERACTIVE TIME-SERIES EXPLORER

**Purpose:** Interactive exploration of time series patterns by groups

In [ ]:
# ============================================
# INTERACTIVE TIME-SERIES EXPLORER
# ============================================
print("="*60)
print("INTERACTIVE TIME-SERIES EXPLORER")
print("="*60)

# Get unique values for dropdowns
unique_codes = train_sample['code'].unique().to_list()
unique_sub_codes = train_sample['sub_code'].unique().to_list()
unique_sub_categories = train_sample['sub_category'].unique().to_list()
unique_horizons = sorted(train_sample['horizon'].unique().to_list())

# Create widgets
group_by_dropdown = Dropdown(
    options=['code', 'sub_code', 'sub_category'],
    value='code',
    description='Group by:',
    style={'description_width': 'initial'}
)

group_value_dropdown = Dropdown(
    options=unique_codes,
    value=unique_codes[0] if unique_codes else None,
    description='Value:',
    style={'description_width': 'initial'}
)

horizon_ts_dropdown = Dropdown(
    options=unique_horizons,
    value=unique_horizons[0] if unique_horizons else None,
    description='Horizon:',
    style={'description_width': 'initial'}
)

max_points_slider = IntSlider(
    min=100,
    max=5000,
    value=1000,
    step=100,
    description='Max Points:',
    style={'description_width': 'initial'}
)

output_ts_area = Output()

def update_group_values(change=None):
    group_by = group_by_dropdown.value
    if group_by == 'code':
        group_value_dropdown.options = unique_codes
    elif group_by == 'sub_code':
        group_value_dropdown.options = unique_sub_codes
    elif group_by == 'sub_category':
        group_value_dropdown.options = unique_sub_categories

def plot_time_series(change=None):
    with output_ts_area:
        output_ts_area.clear_output()
        
        group_by = group_by_dropdown.value
        group_value = group_value_dropdown.value
        horizon = horizon_ts_dropdown.value
        max_points = max_points_slider.value
        
        if group_value is None or horizon is None:
            print("Please select valid options")
            return
        
        # Filter data
        filtered = train_sample.filter(
            (pl.col(group_by) == group_value) & 
            (pl.col('horizon') == horizon)
        ).sort('ts_index')
        
        if len(filtered) == 0:
            print(f"No data for {group_by}={group_value}, horizon={horizon}")
            return
        
        # Limit points for performance
        if len(filtered) > max_points:
            step = len(filtered) // max_points
            filtered = filtered.filter(
                pl.col('ts_index') % step == 0
            )
        
        # Extract data
        ts_index = filtered['ts_index'].to_numpy()
        y_target = filtered['y_target'].to_numpy()
        weight = filtered['weight'].to_numpy()
        
        # Create plot
        fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 12))
        
        # Target vs time
        ax1.plot(ts_index, y_target, 'b-', alpha=0.7, linewidth=1)
        ax1.set_title(f'y_target vs Time\n{group_by}={group_value}, horizon={horizon} (n={len(y_target)})')
        ax1.set_xlabel('Time Index')
        ax1.set_ylabel('y_target')
        ax1.grid(True, alpha=0.3)
        
        # Weight vs time
        ax2.plot(ts_index, weight, 'r-', alpha=0.7, linewidth=1)
        ax2.set_title(f'Weight vs Time\n{group_by}={group_value}, horizon={horizon}')
        ax2.set_xlabel('Time Index')
        ax2.set_ylabel('Weight')
        ax2.grid(True, alpha=0.3)
        
        # Scatter plot: target vs weight
        scatter = ax3.scatter(weight, y_target, alpha=0.6, s=20)
        ax3.set_title(f'y_target vs Weight\n{group_by}={group_value}, horizon={horizon}')
        ax3.set_xlabel('Weight')
        ax3.set_ylabel('y_target')
        ax3.grid(True, alpha=0.3)
        
        # Add correlation info
        if len(y_target) > 1 and len(weight) > 1:
            corr = np.corrcoef(y_target, weight)[0, 1]
            ax3.text(0.02, 0.98, f'Correlation: {corr:.4f}', 
                    transform=ax3.transAxes, fontsize=10,
                    verticalalignment='top',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        plt.tight_layout()
        plt.show()

# Connect widgets
group_by_dropdown.observe(update_group_values, names='value')
group_by_dropdown.observe(plot_time_series, names='value')
group_value_dropdown.observe(plot_time_series, names='value')
horizon_ts_dropdown.observe(plot_time_series, names='value')
max_points_slider.observe(plot_time_series, names='value')

# Display widgets
ts_controls = VBox([HBox([group_by_dropdown, group_value_dropdown]), 
                    HBox([horizon_ts_dropdown, max_points_slider])])
display(ts_controls, output_ts_area)

# Initial plot
plot_time_series()

## STEP 2: MISSING VALUES ANALYSIS

**Purpose:** Analyze missing patterns and their impact on target/weight

In [ ]:
# ============================================
# MISSING VALUES ANALYSIS
# ============================================
print("="*60)
print("MISSING VALUES ANALYSIS")
print("="*60)

# Check missing values
missing_analysis = []
for col in train_sample.columns:
    null_count = train_sample[col].null_count()
    null_pct = (null_count / len(train_sample)) * 100
    dtype = train_sample[col].dtype
    
    missing_analysis.append({
        'column': col,
        'null_count': null_count,
        'null_percentage': null_pct,
        'dtype': dtype
    })

missing_df = pl.DataFrame(missing_analysis).sort('null_count', descending=True)

print("MISSING VALUES SUMMARY:")
print(missing_df)

# Features with missing values
missing_features = missing_df.filter(
    (pl.col('null_count') > 0) & 
    (pl.col('column').str.starts_with('feature_'))
)

if len(missing_features) > 0:
    print(f"\nFEATURES WITH MISSING VALUES: {len(missing_features)}")
    for row in missing_features.iter_rows():
        col, count, pct, dtype = row
        print(f"  {col:20s}: {count:6,} ({pct:5.2f}%) - {dtype}")
        
    # Check which groups have missing features
    print("\nMISSING FEATURES BY GROUP:")
    missing_feature_names = missing_features['column'].to_list()
    
    for group_name, group_features in FEATURE_GROUPS.items():
        if group_features:
            missing_in_group = [f for f in group_features if f in missing_feature_names]
            if missing_in_group:
                print(f"  {group_name.upper():8s}: {missing_in_group}")
else:
    print("\nNo missing values in features (sample data)")

## PROJECT INTEGRATION SUMMARY

**How this analysis integrates with your project:**

### **1. Data Quality Validation**
- Ensures data is clean before LGBM SHAP training
- Identifies missing patterns that might affect feature engineering
- Validates feature groups (ULTRA/HIGH/CORE) integrity

### **2. Feature Engineering Pipeline**
- Checks if SHAP features exist and have proper distributions
- Validates BNN aggregated features creation
- Ensures engineered features are within expected ranges

### **3. Model Training Preparation**
- Confirms data structure matches model expectations
- Identifies potential data leakage issues
- Validates target/weight distributions

### **4. Performance Considerations**
- Sample-based widgets for interactive exploration
- Full dataset analysis available in separate cells
- Memory-efficient operations with Polars

### **5. Scientific Rigor**
- Comprehensive statistical analysis
- Multiple visualization perspectives
- Reproducible analysis pipeline

**Next Steps:**
1. Run full dataset analysis (not just sample)
2. Add correlation analysis (Spearman/Pearson)
3. Implement outlier detection
4. Add feature importance analysis
5. Create automated sanity check reports